In [ ]:
!git clone https://github.com/ODBapp/2026_NODASS_workshop.git

In [ ]:
%cd 2026_NODASS_workshop/src/geo

# 地形資料

1. **NetCDF檔**
2. **ODB GEBCO API**
3. **網格檔**
4. **GEBCO vs ETOPO**

In [ ]:
%pip install -q netCDF4

## 準備

In [ ]:
import json
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import netCDF4
import rasterio
from rasterio.transform import from_origin

W, E, S, N = 120.0, 122.0, 22.0, 23.5
ETOPO_FILE = Path("ETOPO_2022_v1_15s_N30E120_surface.nc")
GEBCO_CSV = Path("gebco_odb_sample.csv")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)
ASPECT = 1 / np.cos(np.deg2rad((S + N) / 2)) # 修正地圖變形

print(f"ETOPO 檔案就緒：{ETOPO_FILE.name}({ETOPO_FILE.stat().st_size/1e6:.1f} MB)")

## NetCDF 檔案

NetCDF檔案會記載其metadata，使用前務必先查閱。

In [ ]:
nc = netCDF4.Dataset(ETOPO_FILE)
nc   # 直接執行

從dimensions可以得知這個檔案可以組成3600 * 3600的二維平面  
其中變數有crs(無維度純描述)、經度(長度3600陣列)、緯度(長度3600陣列)、z(3600 * 3600陣列)

雖然知道經緯度的數量，但我們不知道經緯度的範圍

In [ ]:
nc_lat = nc.variables["lat"][:]
nc_lon = nc.variables["lon"][:]
print(f"nc 緯度最小值: {np.min(nc_lat)}")
print(f"nc 緯度最大值: {np.max(nc_lat)}")
print(f"nc 經度最小值: {np.min(nc_lon)}")
print(f"nc 經度最大值: {np.max(nc_lon)}")

由此可知這個.nc的地圖範圍是120 ~ 135E 15 ~ 30N  
15 / 3600 = 0.00416667度 網格大小也就是15*15弧秒(3600弧秒 = 1度)

**接著看變數z的內容**

In [ ]:
nc.variables["z"]

變數z是代表高度(height)、以EGM2008 (全球通用geoid)為基準，是我們要找的高程資料  
缺值是-99999.0、單位是公尺、向上為正值

從netCDF切出需要的範圍(120 ~ 122.0E、22 ~ 23.5N)，減少記憶體用量

In [ ]:
lon_full = np.ma.filled(nc_lon, np.nan) # 整理無效值
lat_full = np.ma.filled(nc_lat, np.nan)
i0, i1 = np.searchsorted(lon_full, [W, E]) # 找出座標在陣列的位置
j0, j1 = np.searchsorted(lat_full, [S, N])
e_lon, e_lat = lon_full[i0:i1], lat_full[j0:j1] # 取得該位置實際的經緯度座標值
e_z = np.ma.filled(nc.variables["z"][j0:j1, i0:i1], np.nan).astype("float64") # 取得此經緯度範圍內的z值

# 關檔前先記下垂直基準（後面和 GEBCO 比較時會用到）
ETOPO_VERT = getattr(nc.variables["z"], "vert_crs_name", "未標示")
ETOPO_VERT_EPSG = getattr(nc.variables["z"], "vert_crs_epsg", "未標示")
nc.close()

print(f"高程 {np.nanmin(e_z):.0f} ~ {np.nanmax(e_z):.0f} m")

## ODB GEBCO API

API 就像向資料庫下訂單：URL 參數＝訂單內容，伺服器收單後出貨。  
參數內容請參考每個API的說明

| 訂單欄位 | 意義 |
|---|---|
| `mode=truncate,zonly` | truncate: 裁到小數點後5位; zonly: 只回傳z值和經緯度 |
| `sample=1` | n => n/240 |
| `jsonsrc={GeoJSON}` | 用 GeoJSON描述範圍 |

回傳為 JSON（longitude/latitude/z 三個陣列），把他轉成CSV，excel也可以讀

In [ ]:
polygon = [[E, N], [W, N], [W, S], [E, S]] # 多邊形點位
jsonsrc = json.dumps({"type": "Polygon", "coordinates": [polygon]}) # 組合成json
url = ("https://service.oc.ntu.edu.tw/data/gebco?" +
       urllib.parse.urlencode({"mode": "truncate,zonly", "sample": "1",
                               "jsonsrc": jsonsrc})) # API網址 aka 訂單
print(f"也可以直接輸入網址 {url}")

with urllib.request.urlopen(url, timeout=600) as resp: # 發送請求(訂單)到資料伺服器
    payload = resp.read().decode("utf-8") # 回傳的資料


obj = json.loads(payload) # json轉為python object
pts = np.column_stack([obj["longitude"], obj["latitude"], obj["z"]]) # 轉為三欄的表格
np.savetxt(GEBCO_CSV, pts, delimiter=",", fmt=["%.5f", "%.5f", "%.2f"],
           header="Longitude,Latitude,Elevations(m)", comments="")

print(f"已轉存 {GEBCO_CSV.name}({pts.shape[0]} 點，"
      f"{GEBCO_CSV.stat().st_size/1e6:.1f} MB)，前 3 列：")

for row in pts[:3]:
    print(f"  {row[0]:.5f}, {row[1]:.5f}, {row[2]:.2f}")

## 網格檔

### CSV 點資料 → 2D 網格

CSV 一列一點，是「**未網格化**」的資料

In [ ]:
lon_pts, lat_pts, z_pts = np.loadtxt(GEBCO_CSV, delimiter=",", skiprows=1).T # 讀CSV檔案
print(f"經度 {lon_pts.min():.4f} - {lon_pts.max():.4f}，"
      f"緯度 {lat_pts.min():.4f} - {lat_pts.max():.4f}")

def validate_axis(label, values):
    """檢查座標軸能不能拿來比對網格，回傳轉好型別的陣列。

    只擋兩種這裡真的會遇到的情況：格點太少（範圍縮太小，或 API 回了
    空結果），以及座標含 NaN（來源檔有遮罩時會出現）。不擋的話，
    後面只會得到看不懂的 IndexError。
    """
    a = np.asarray(values, dtype=np.float64)
    if a.size < 2:
        raise ValueError(f"{label}只有 {a.size} 個格點，至少要兩個才能判斷間距與方向。"
                         "請把 W/E/S/N 的範圍拉開一點再試。")
    if not np.all(np.isfinite(a)):
        raise ValueError(f"{label}含有 NaN，無法用來比對網格")
    return a


def regular_axis(c, tol=0.05):
    """驗證座標軸為等距網格，取得兩軸座標"""
    c = validate_axis("座標軸", c)
    d = (c[-1] - c[0]) / (c.size - 1)
    reg = c[0] + d * np.arange(c.size)
    if np.abs(c - reg).max() > tol * abs(d):
        raise ValueError("座標不是等距網格，需先重新取樣")
    return reg, d

g_lon, dg_lon = regular_axis(np.unique(lon_pts))
g_lat, dg_lat = regular_axis(np.unique(lat_pts))
print(f"唯一經度 {g_lon.size} 個、唯一緯度 {g_lat.size} 個，"
      f"間距 {dg_lon*3600:.1f}″×{dg_lat*3600:.1f}″ ")

g_z = np.full((g_lat.size, g_lon.size), np.nan) # 空白陣列
ix = np.round((lon_pts - g_lon[0]) / dg_lon).astype(int)   # 每點的行/列索引
iy = np.round((lat_pts - g_lat[0]) / dg_lat).astype(int)
g_z[iy, ix] = z_pts # 把 1×172800 的序列依索引填入 480×360 網格

print(f"完成：{g_lon.size}×{g_lat.size} 網格，"
      f"高程 {np.nanmin(g_z):.0f} ~ {np.nanmax(g_z):.0f} m")

繪製地圖

In [ ]:
from matplotlib.colors import TwoSlopeNorm

def edges_from_centers(c):
    """繪圖用，找出格子邊界(n+1條)座標"""
    mid = (c[:-1] + c[1:]) / 2
    return np.concatenate([[c[0] - (c[1] - c[0]) / 2], mid,
                           [c[-1] + (c[-1] - c[-2]) / 2]])

norm = TwoSlopeNorm(vmin=min(np.nanmin(e_z), np.nanmin(g_z)), vcenter=0.0,
                    vmax=max(np.nanmax(e_z), np.nanmax(g_z))) # 保持0在中間，左右兩側依最大最小值縮放
fig, axes = plt.subplots(1, 2, figsize=(10.5, 6.2), sharey=True,
                         constrained_layout=True)

for ax, (zz, lon, lat, title) in zip(axes, [
        (e_z, e_lon, e_lat, "ETOPO 2022 (NetCDF)"),
        (g_z, g_lon, g_lat, "GEBCO (ODB API CSV)")]):

    im = ax.pcolormesh(edges_from_centers(lon), edges_from_centers(lat),
                       np.ma.masked_invalid(zz), cmap="BrBG_r", norm=norm,
                       shading="flat") #網格地圖，需要知道邊緣而不是網格中心座標
    ax.set_aspect(ASPECT)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Longitude (°E)", fontsize=9)
    ax.tick_params(labelsize=8)
axes[0].set_ylabel("Latitude (°N)", fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.8, label="Elevation (m)")
plt.show()

### 輸出 GeoTIFF 與 .asc
.asc雖然有標示角落座標，但沒有CRS，還是需要投影資訊(.prj)才能放到GIS軟體  
且.asc只有單一cellsize，僅適用正方形網格

In [ ]:
NODATA = -9999.0

def write_asc(arr, lon, lat, path, fmt="%.1f"):
    """輸出.asc＋ .prj（WGS84）。"""
    dx = float(np.diff(lon).mean())
    dy = float(np.diff(lat).mean())

    cell = (dx + dy) / 2
    data = np.flipud(np.where(np.isfinite(arr), arr, NODATA))
    with open(path, "w", newline="\n") as f:
        f.write(f"ncols {data.shape[1]}\n"
                f"nrows {data.shape[0]}\n"
                f"xllcorner {lon[0] - cell / 2:.10g}\n"
                f"yllcorner {lat[0] - cell / 2:.10g}\n"
                f"cellsize {cell:.10g}\n"
                f"NODATA_value {NODATA:g}\n") # 寫入.asc標準六行表頭
        np.savetxt(f, data, fmt=fmt)

    # 輸出.prj
    WKT_WGS84 = ('GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]]')
    Path(path).with_suffix(".prj").write_text(WKT_WGS84)
    print(f"  已輸出 {path}({data.shape[1]}×{data.shape[0]})")


def write_geotiff(arr, lon, lat, path):
    """輸出 GeoTIFF"""
    dx = float(np.diff(lon).mean())
    dy = float(np.diff(lat).mean())
    transform = from_origin(lon[0] - dx / 2, lat[-1] + dy / 2, dx, dy)  # 左上角
    data = np.flipud(np.where(np.isfinite(arr), arr, NODATA)).astype("float32")
    with rasterio.open(path, "w", driver="GTiff",
                       height=data.shape[0], width=data.shape[1], count=1,
                       dtype="float32", crs="EPSG:4326", transform=transform,
                       nodata=NODATA, compress="deflate") as dst:
        dst.write(data, 1)
    print(f"  已輸出 {path}({data.shape[1]}×{data.shape[0]})")

write_asc(e_z, e_lon, e_lat, OUT_DIR / "etopo2022_elev_15s.asc")
write_geotiff(e_z, e_lon, e_lat, OUT_DIR / "etopo2022_elev_15s.tif")
write_asc(g_z, g_lon, g_lat, OUT_DIR / "gebco_elev_15s.asc")
write_geotiff(g_z, g_lon, g_lat, OUT_DIR / "gebco_elev_15s.tif")

## GEBCO vs ETOPO
高程直接相減的前提是兩網格「形狀相同且重合」  
若改變範圍或cell size，必須先對齊才能相減

除了水平方向要對齊，**垂直基準**也要一致（呼應講義「地圖及高程」一節：一切都是相對的）：

| | 水平基準 | 垂直基準 |
|---|---|---|
| ETOPO 2022 | WGS84 (EPSG:4326) | EGM2008 大地水準面 (EPSG:3855)，檔案裡的 `z:vert_crs_name` 有標示 |
| GEBCO（ODB API） | WGS84 | 官方說明是「假設所有來源資料皆以平均海水面(MSL)為準」，但也指出「部分淺水區的來源資料使用了不同的垂直基準」 |

> ODB 的 GEBCO endpoint 沒有版本欄位，回應只有經緯度與 z。工作坊標示為 2026 Grid，
> 但這一點無法由 API 回應本身確認，引用時要說明來源與取得日期。

所以下面算出來的差值，除了兩個地形模型本身的差異，也混進了垂直基準的差異。
這種比較叫做 **intercomparison（互相比對）**，不是 accuracy validation（精度驗證），
因為兩者都不是實測真值，誰也不能當作標準答案。

GEBCO 的警告特別指向**淺水區**，而淺水區的高程差本來就比深海小得多，
所以基準差異在那裡佔的比重反而最高。看下面的圖時，近岸和外海要分開解讀。

GEBCO 垂直基準說明：<https://www.gebco.net/data-products-gridded-bathymetry-data/gebco2026-grid>


In [ ]:
# 相減之前，先確認兩個網格真的指向同一批格點。
# 兩個都是 360×480，形狀一樣不代表可以相減：可能差半格、南北顛倒，或垂直基準不同。

def check_before_subtract(name_a, lon_a, lat_a, name_b, lon_b, lat_b,
                          quantize_atol=1e-5, max_cell_frac=0.01):
    """逐項比對兩個網格並印出結果，對不上就擋下來，不算出沒有意義的差值。

    座標為什麼不能用 == 直接比？ODB GEBCO API 回傳的座標已量化到小數第 5 位
    （120.00208 對應的精確格心是 120.0020833...），差約 3e-6 度。那是數值表示
    造成的，不是網格對不上。所以容許值取「量化上限 1e-5 度」與「一格的 1%」
    之中較嚴格的那個，格距變大時才不會跟著放寬。
    """
    lon_a = validate_axis(f"{name_a} 經度", lon_a)
    lat_a = validate_axis(f"{name_a} 緯度", lat_a)
    lon_b = validate_axis(f"{name_b} 經度", lon_b)
    lat_b = validate_axis(f"{name_b} 緯度", lat_b)

    print(f"{name_a}  vs  {name_b}")
    problems = []

    # 1. 形狀
    shape_a, shape_b = (lat_a.size, lon_a.size), (lat_b.size, lon_b.size)
    same_shape = shape_a == shape_b
    print(f"  形狀      {shape_a} vs {shape_b}  ->  {'一致' if same_shape else '不一致'}")
    if not same_shape:
        problems.append("形狀不同")

    # 2. 座標逐點比對（形狀一樣才比得下去）
    #    經度一度的地面距離會隨緯度縮短，換算公尺時要乘上 cos(緯度)，
    #    和前面算 ASPECT 修正地圖變形是同一個道理
    mid_lat = float((lat_a.mean() + lat_b.mean()) / 2)
    m_per_deg = {"經度": 111320 * np.cos(np.deg2rad(mid_lat)), "緯度": 111320}
    if same_shape:
        for axis, a, b in (("經度", lon_a, lon_b), ("緯度", lat_a, lat_b)):
            step = float(np.median(np.abs(np.diff(a))))   # 一格有多寬，取中位數較穩健
            if step <= 0:
                raise ValueError(f"{axis}的格距為 0，座標可能有重複值")
            allowed = min(quantize_atol, max_cell_frac * step)
            dmax = float(np.abs(a - b).max())             # 最大偏移
            ok = dmax <= allowed
            print(f"  {axis}      最大差 {dmax:.2e}° = {dmax / step:.4f} 格"
                  f"（約 {dmax * m_per_deg[axis]:.2f} m，容許 {allowed:.2e}°）"
                  f"  ->  {'一致' if ok else '不一致'}")
            if not ok:
                problems.append(f"{axis}對不上，常見原因是一邊用格子中心、另一邊用格子角落")

    # 3. 軸的方向：NetCDF 的緯度不一定是遞增，方向相反會讓結果南北顛倒
    for axis, a, b in (("經度", lon_a, lon_b), ("緯度", lat_a, lat_b)):
        dir_a = "遞增" if a[1] > a[0] else "遞減"
        dir_b = "遞增" if b[1] > b[0] else "遞減"
        ok = dir_a == dir_b
        print(f"  {axis}方向  {dir_a} vs {dir_b}  ->  {'一致' if ok else '相反'}")
        if not ok:
            problems.append(f"{axis}方向相反")

    if problems:
        raise ValueError("網格不一致，不能直接相減：" + "；".join(problems))


check_before_subtract("ETOPO 2022", e_lon, e_lat, "GEBCO (ODB API)", g_lon, g_lat)

# 水平方向查完了，垂直基準沒辦法從座標看出來，要看各自的說明文件
print(f"  垂直基準  ETOPO: {ETOPO_VERT} ({ETOPO_VERT_EPSG})")
print( "            GEBCO: 官方假設為平均海水面(MSL)，部分淺水區來源不同  ->  不一致")
print( "            兩者的差值因此也包含基準差異，屬於互相比對而非精度驗證")


In [ ]:
diff = g_z - e_z # 高度差（m）：正 = GEBCO 較高
ok = np.isfinite(diff)

vlim = max(50.0, float(np.ceil(np.nanpercentile(np.abs(diff[ok]), 99) / 50) * 50))
fig, (axm, axh) = plt.subplots(1, 2, figsize=(12.0, 6.2),
                               width_ratios=[1.0, 1.3], constrained_layout=True)
im = axm.pcolormesh(edges_from_centers(g_lon), edges_from_centers(g_lat),
                    np.ma.masked_invalid(diff), cmap="RdBu_r",
                    vmin=-vlim, vmax=vlim, shading="flat")
axm.contour(g_lon, g_lat, np.where(np.isfinite(e_z), e_z, 0),
            levels=[0], colors="#333333", linewidths=0.7)   # coastline (z = 0)
axm.set_aspect(ASPECT)
axm.set_title("GEBCO - ETOPO (m)", fontsize=11)
axm.set_xlabel("Longitude (°E)", fontsize=9)
axm.set_ylabel("Latitude (°N)", fontsize=9)
axm.tick_params(labelsize=8)
fig.colorbar(im, ax=axm, shrink=0.9, label="Difference (m)")

# 高差分布圖
# 統計數據
n = int(ok.sum())
bias = float(np.nanmean(diff))
rmse = float(np.sqrt(np.nanmean(diff[ok] ** 2)))
pct10 = float((np.abs(diff[ok]) <= 10).mean() * 100)
med = float(np.nanmedian(diff))

axh.hist(diff[ok], bins=np.linspace(-vlim, vlim, 81), color="#4878a8")
axh.set_yscale("log")
axh.axvline(0, color="#666666", linewidth=0.8, linestyle="--")
axh.set_title("Distribution (log scale)", fontsize=11)
axh.set_xlabel("Difference (m)", fontsize=9)
axh.set_ylabel("Count", fontsize=9)
axh.tick_params(labelsize=8)
axh.text(0.02, 0.97,
         f"n = {n}\nmean = {bias:+.1f} m\nmedian = {med:+.1f} m\nRMSE = {rmse:.1f} m\n"
         f"within \u00b110 m: {pct10:.1f}%",
         transform=axh.transAxes, va="top", fontsize=9.5,
         bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"))
plt.show()